# Lesson 5A: Principal Component Analysis — Theory and Mathematical Foundations

---

## Overview

**Principal Component Analysis (PCA)** is the most fundamental technique for dimensionality reduction in machine learning. It transforms high-dimensional data into a lower-dimensional representation that preserves maximum variance, enabling visualization, noise reduction, and computational efficiency.

**Historical Context**: PCA was independently developed by Karl Pearson (1901) for biological sciences and Harold Hotelling (1933) for economics. It remains one of the most widely used statistical techniques today.

**Key References**:
- Pearson, K. (1901). "On Lines and Planes of Closest Fit to Systems of Points in Space". *Philosophical Magazine*, 2(11), 559-572.
- Hotelling, H. (1933). "Analysis of a Complex of Statistical Variables into Principal Components". *Journal of Educational Psychology*, 24(6), 417-441.
- Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer.
- Tipping, M. E., & Bishop, C. M. (1999). "Probabilistic Principal Component Analysis". *Journal of the Royal Statistical Society: Series B*, 61(3), 611-622.

---

## Table of Contents

1. Motivation and the Curse of Dimensionality
2. Problem Formulation
3. Variance Maximization Derivation
4. Eigendecomposition Approach
5. Singular Value Decomposition (SVD) Approach
6. Geometric Interpretation
7. Choosing the Number of Components
8. Computational Complexity
9. Variants and Extensions
10. Limitations

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_iris, fetch_olivetti_faces
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist
import seaborn as sns

# Visualization configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

# Reproducibility
np.random.seed(42)

print("Environment configured successfully")
print(f"NumPy version: {np.__version__}")

---

## 1. Motivation and the Curse of Dimensionality

### 1.1 The High-Dimensional Data Problem

Modern datasets often contain hundreds or thousands of features:
- **Genomics**: 20,000+ gene expression levels per sample
- **Computer Vision**: 784 pixels for 28×28 images, millions for high-resolution
- **Text**: Vocabulary sizes of 50,000+ words
- **Sensor Networks**: Hundreds of simultaneous measurements

### 1.2 The Curse of Dimensionality

**Definition** (Bellman, 1961): As dimensionality increases, several pathological phenomena emerge that fundamentally change data behavior.

#### 1.2.1 Volume Concentration

In high dimensions, volume concentrates in the corners of a hypercube.

**Theorem**: For a unit hypercube in $d$ dimensions, the ratio of the volume of an inscribed sphere to the hypercube volume is:

$$\frac{V_{\text{sphere}}(d)}{V_{\text{cube}}(d)} = \frac{\pi^{d/2}}{2^d \Gamma(d/2 + 1)} \xrightarrow{d \to \infty} 0$$

**Consequence**: In $d=10$ dimensions, the inscribed sphere occupies only 0.25% of the cube's volume!

#### 1.2.2 Distance Concentration

**Theorem** (Beyer et al., 1999): For i.i.d. random variables with finite variance, the ratio of distances to the nearest and farthest neighbors converges to 1:

$$\lim_{d \to \infty} \mathbb{E}\left[\frac{D_{\max}(d) - D_{\min}(d)}{D_{\min}(d)}\right] = 0$$

**Implication**: Distance-based methods (k-NN, k-means) become ineffective.

#### 1.2.3 Sample Complexity

To maintain constant data density, required samples grow exponentially:

$$n_{\text{required}} \sim c^d$$

**Example**: If 100 samples suffice in 1D, you need $100^{10} = 10^{20}$ samples in 10D!

### 1.3 Why Dimensionality Reduction?

**Benefits**:
1. **Visualization**: Project to 2D or 3D for human interpretation
2. **Computational Efficiency**: Reduce storage and computation time
3. **Noise Reduction**: Remove less important (noisy) dimensions
4. **Feature Engineering**: Create uncorrelated features for downstream tasks
5. **Regularization**: Prevent overfitting in high-dimensional regression/classification

---

## 2. Problem Formulation

### 2.1 Formal Setup

**Input**:
- Data matrix $X \in \mathbb{R}^{m \times n}$ with $m$ samples and $n$ features
- Target dimensionality $k$ where $k \ll n$

**Output**:
- Projection matrix $W \in \mathbb{R}^{n \times k}$
- Projected data $Z = XW \in \mathbb{R}^{m \times k}$

**Objective**: Find $W$ such that projected data preserves maximum variance (information).

### 2.2 Assumptions

1. **Linearity**: PCA assumes a linear subspace captures the data
2. **Gaussian-like**: Works best when data is approximately Gaussian
3. **Variance = Information**: High variance directions are most informative
4. **Orthogonality**: Principal components are mutually orthogonal

---

## 3. Variance Maximization Derivation

### 3.1 First Principal Component

**Goal**: Find unit vector $\mathbf{w}_1$ that maximizes projected variance.

For centered data $\bar{X} = X - \mathbb{E}[X]$, the variance of projections onto $\mathbf{w}_1$ is:

$$\text{Var}(\bar{X}\mathbf{w}_1) = \frac{1}{m}\|\bar{X}\mathbf{w}_1\|^2 = \frac{1}{m}\mathbf{w}_1^T\bar{X}^T\bar{X}\mathbf{w}_1 = \mathbf{w}_1^T C \mathbf{w}_1$$

where $C = \frac{1}{m}\bar{X}^T\bar{X}$ is the covariance matrix.

**Optimization Problem**:
$$\max_{\mathbf{w}_1} \mathbf{w}_1^T C \mathbf{w}_1 \quad \text{subject to} \quad \|\mathbf{w}_1\|^2 = 1$$

**Solution via Lagrange Multipliers**:

Form the Lagrangian:
$$\mathcal{L}(\mathbf{w}_1, \lambda) = \mathbf{w}_1^T C \mathbf{w}_1 - \lambda(\mathbf{w}_1^T\mathbf{w}_1 - 1)$$

Taking derivative and setting to zero:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}_1} = 2C\mathbf{w}_1 - 2\lambda\mathbf{w}_1 = 0$$

This yields the eigenvalue equation:
$$C\mathbf{w}_1 = \lambda\mathbf{w}_1$$

**Conclusion**: $\mathbf{w}_1$ is an eigenvector of $C$, and the maximized variance is the corresponding eigenvalue $\lambda$.

To maximize variance, choose the eigenvector with the **largest eigenvalue**.

### 3.2 Subsequent Principal Components

**Theorem**: The $k$-th principal component is the eigenvector of $C$ with the $k$-th largest eigenvalue, subject to orthogonality with previous components.

**Proof Sketch**:
- Require $\mathbf{w}_k \perp \mathbf{w}_1, \ldots, \mathbf{w}_{k-1}$
- Maximize $\mathbf{w}_k^T C \mathbf{w}_k$ with orthogonality constraints
- Solution: Eigenvector with next largest eigenvalue

**Result**: Principal components are the eigenvectors of the covariance matrix, ordered by descending eigenvalues.

---

## 4. Eigendecomposition Approach

### 4.1 Covariance Matrix

For centered data $\bar{X} \in \mathbb{R}^{m \times n}$:

$$C = \frac{1}{m}\bar{X}^T\bar{X} \in \mathbb{R}^{n \times n}$$

**Properties**:
- **Symmetric**: $C^T = C$
- **Positive Semi-Definite**: $\mathbf{v}^T C \mathbf{v} \geq 0$ for all $\mathbf{v}$
- **Real Eigenvalues**: All eigenvalues $\lambda_i \in \mathbb{R}$
- **Orthogonal Eigenvectors**: $\mathbf{w}_i^T \mathbf{w}_j = \delta_{ij}$

### 4.2 Spectral Theorem

For symmetric matrix $C$:

$$C = Q\Lambda Q^T$$

where:
- $Q = [\mathbf{w}_1 | \mathbf{w}_2 | \cdots | \mathbf{w}_n] \in \mathbb{R}^{n \times n}$ (orthogonal matrix of eigenvectors)
- $\Lambda = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_n)$ (diagonal matrix of eigenvalues)
- $Q^TQ = QQ^T = I$

**Ordering**: $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_n \geq 0$

### 4.3 PCA Algorithm (Eigendecomposition)

```
Input: Data matrix X ∈ ℝ^{m×n}, target dimension k

1. Center data:
   μ ← (1/m) Σ_{i=1}^m x_i
   X̄ ← X - μ

2. Compute covariance matrix:
   C ← (1/m) X̄^T X̄

3. Eigendecomposition:
   Compute eigenvectors w_i and eigenvalues λ_i of C

4. Sort by eigenvalue:
   Order (w_i, λ_i) by decreasing λ_i

5. Select top-k:
   W ← [w_1 | w_2 | ... | w_k] ∈ ℝ^{n×k}

6. Project data:
   Z ← X̄ W ∈ ℝ^{m×k}

Output: Projection matrix W, projected data Z
```

**Computational Complexity**: $O(n^3)$ for eigendecomposition (dominated by Step 3)

---

## 5. Singular Value Decomposition (SVD) Approach

### 5.1 SVD Formulation

Every matrix $\bar{X} \in \mathbb{R}^{m \times n}$ has a SVD:

$$\bar{X} = U\Sigma V^T$$

where:
- $U \in \mathbb{R}^{m \times m}$: Left singular vectors (orthogonal)
- $\Sigma \in \mathbb{R}^{m \times n}$: Singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ on diagonal
- $V \in \mathbb{R}^{n \times n}$: Right singular vectors (orthogonal)

### 5.2 Connection to PCA

**Theorem**: The right singular vectors of $\bar{X}$ are the eigenvectors of $C = \frac{1}{m}\bar{X}^T\bar{X}$.

**Proof**:
$$C = \frac{1}{m}\bar{X}^T\bar{X} = \frac{1}{m}(U\Sigma V^T)^T(U\Sigma V^T) = \frac{1}{m}V\Sigma^T U^T U \Sigma V^T = \frac{1}{m}V\Sigma^2 V^T$$

Thus $V$ contains eigenvectors of $C$, and $\lambda_i = \sigma_i^2 / m$.

### 5.3 PCA via SVD

```
Input: Centered data X̄ ∈ ℝ^{m×n}, target dimension k

1. Compute SVD:
   X̄ = UΣV^T

2. Extract principal components:
   W ← First k columns of V

3. Project data:
   Z ← X̄W = UΣV^T V[:,:k] = U[:,:k] Σ[:k,:k]

Output: W, Z
```

**Advantages over Eigendecomposition**:
- More numerically stable
- Faster when $m \ll n$ (compute truncated SVD)
- Directly provides projected data via $U$ and $\Sigma$

**Computational Complexity**:
- Full SVD: $O(\min(mn^2, m^2n))$
- Truncated SVD (randomized): $O(mnk)$ where $k \ll \min(m,n)$

---

## 6. Geometric Interpretation

### 6.1 Change of Basis

PCA performs a **rotation** of the coordinate system:
- Original axes: Standard basis $\{\mathbf{e}_1, \ldots, \mathbf{e}_n\}$
- New axes: Principal components $\{\mathbf{w}_1, \ldots, \mathbf{w}_n\}$

The new axes are:
1. **Aligned with maximum variance directions**
2. **Mutually orthogonal**
3. **Ordered by decreasing variance**

### 6.2 Projection and Reconstruction

**Projection** onto $k$ components:
$$\mathbf{z}_i = W^T(\mathbf{x}_i - \boldsymbol{\mu}), \quad \mathbf{z}_i \in \mathbb{R}^k$$

**Reconstruction** from $k$ components:
$$\hat{\mathbf{x}}_i = W\mathbf{z}_i + \boldsymbol{\mu}$$

**Reconstruction Error**:
$$\text{MSE} = \frac{1}{m}\sum_{i=1}^m \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2 = \sum_{j=k+1}^n \lambda_j$$

The error equals the sum of discarded eigenvalues!

### 6.3 Best Linear Approximation

**Theorem** (Eckart-Young): PCA provides the best rank-$k$ linear approximation of the data in terms of squared Frobenius norm:

$$W^* = \arg\min_{W: \text{rank}(W)=k} \|X - XWW^T\|_F^2$$

---

## 7. Choosing the Number of Components

### 7.1 Explained Variance Ratio

The variance explained by the first $k$ components:

$$\text{EVR}(k) = \frac{\sum_{i=1}^k \lambda_i}{\sum_{i=1}^n \lambda_i}$$

**Rule of Thumb**: Choose $k$ such that EVR$(k) \geq 0.90$ or $0.95$ (retain 90-95% of variance).

### 7.2 Scree Plot

Plot eigenvalues $\lambda_i$ vs. component index $i$.

**Look for**: "Elbow" where eigenvalues drop sharply.

### 7.3 Kaiser Criterion

**Rule**: Keep components with $\lambda_i > 1$ (for standardized data).

**Rationale**: Each standardized variable has variance 1; keep components explaining more than a single variable.

### 7.4 Cross-Validation

For supervised tasks:
- Try different values of $k$
- Evaluate downstream performance (classification accuracy, etc.)
- Choose $k$ that maximizes validation performance

---

## 8. Computational Complexity

### 8.1 Time Complexity

| Method | Complexity | When to Use |
|--------|-----------|-------------|
| Covariance + Eigen | $O(n^2m + n^3)$ | $m > n$ (more samples than features) |
| SVD (full) | $O(\min(mn^2, m^2n))$ | General case |
| SVD (truncated) | $O(mnk)$ | $k \ll \min(m,n)$ |
| Randomized SVD | $O(mnk)$ | Large $m,n$; approximate solution OK |

### 8.2 Space Complexity

- Data: $O(mn)$
- Covariance: $O(n^2)$ (can be prohibitive for $n$ large)
- SVD: $O(m + n)$ (stores $U$ and $V$ sparsely)

**Recommendation**: Use SVD (especially truncated/randomized) for large datasets.

---

## 9. Variants and Extensions

### 9.1 Incremental PCA

**Problem**: Standard PCA requires all data in memory.

**Solution** (Ross et al., 2008): Update components incrementally as new data arrives.

**Use Case**: Streaming data, large datasets that don't fit in RAM.

### 9.2 Kernel PCA

**Problem**: PCA is linear; cannot capture nonlinear structure.

**Solution** (Schölkopf et al., 1998): Apply PCA in high-dimensional feature space via kernel trick.

**Kernel Function**: $k(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x})^T \phi(\mathbf{x}')$

**Popular Kernels**:
- RBF: $k(\mathbf{x}, \mathbf{x}') = \exp(-\gamma\|\mathbf{x} - \mathbf{x}'\|^2)$
- Polynomial: $k(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^T\mathbf{x}' + c)^d$

### 9.3 Sparse PCA

**Problem**: Eigenvectors are dense (all features have non-zero loadings).

**Solution** (Zou et al., 2006): Add $\ell_1$ penalty to encourage sparsity.

**Benefit**: Easier interpretation (only subset of features contribute).

### 9.4 Probabilistic PCA (PPCA)

**Reference**: Tipping & Bishop (1999)

**Model**: Assumes data generated from latent factors with Gaussian noise:
$$\mathbf{x} = W\mathbf{z} + \boldsymbol{\mu} + \boldsymbol{\epsilon}, \quad \mathbf{z} \sim \mathcal{N}(0, I), \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, \sigma^2 I)$$

**Advantages**:
- Handles missing data naturally
- Provides likelihood for model selection
- Maximum likelihood solution equals PCA as $\sigma \to 0$

### 9.5 Robust PCA

**Problem**: PCA is sensitive to outliers.

**Solution** (Candès et al., 2011): Decompose data into low-rank + sparse components:
$$X = L + S$$

where $L$ is low-rank (underlying structure) and $S$ is sparse (outliers).

---

## 10. Limitations

### 10.1 Linearity Assumption

PCA assumes a linear subspace captures the data. Fails for:
- **Manifold data**: Swiss roll, s-curve
- **Nonlinear correlations**: XOR pattern

**Solution**: Use Kernel PCA, autoencoders, or manifold learning (t-SNE, UMAP).

### 10.2 Variance ≠ Information

PCA assumes high variance = high information. Fails when:
- **Signal is low-variance**: Discriminative information in small variance directions
- **Noise is high-variance**: Dominant variance due to noise

**Example**: In face recognition, lighting variation (high variance) may obscure identity (low variance).

### 10.3 Interpretability

Principal components are linear combinations of all original features:
$$\text{PC}_1 = 0.43 \cdot \text{Feature}_1 + 0.28 \cdot \text{Feature}_2 + \cdots$$

**Challenge**: Hard to interpret meaning of components.

**Solution**: Use sparse PCA or domain knowledge.

### 10.4 Global Method

PCA finds a single global projection. Fails for:
- **Multiple clusters** with different orientations
- **Hierarchical structure**

**Solution**: Use local methods (LLE, Isomap) or hierarchical approaches.

---

## References

### Foundational Papers

1. **Pearson, K.** (1901). "On Lines and Planes of Closest Fit to Systems of Points in Space". *Philosophical Magazine*, 2(11), 559-572.

2. **Hotelling, H.** (1933). "Analysis of a Complex of Statistical Variables into Principal Components". *Journal of Educational Psychology*, 24(6), 417-441.

### Textbooks

3. **Jolliffe, I. T.** (2002). *Principal Component Analysis* (2nd ed.). Springer.

4. **Bishop, C. M.** (2006). *Pattern Recognition and Machine Learning*. Springer. (Chapter 12: Continuous Latent Variables)

5. **Hastie, T., Tibshirani, R., & Friedman, J.** (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. (Chapter 14.5: Principal Components)

### Extensions and Variants

6. **Tipping, M. E., & Bishop, C. M.** (1999). "Probabilistic Principal Component Analysis". *Journal of the Royal Statistical Society: Series B*, 61(3), 611-622.

7. **Schölkopf, B., Smola, A., & Müller, K. R.** (1998). "Nonlinear Component Analysis as a Kernel Eigenvalue Problem". *Neural Computation*, 10(5), 1299-1319.

8. **Zou, H., Hastie, T., & Tibshirani, R.** (2006). "Sparse Principal Component Analysis". *Journal of Computational and Graphical Statistics*, 15(2), 265-286.

9. **Candès, E. J., Li, X., Ma, Y., & Wright, J.** (2011). "Robust Principal Component Analysis?". *Journal of the ACM*, 58(3), 1-37.

### Computational Methods

10. **Halko, N., Martinsson, P. G., & Tropp, J. A.** (2011). "Finding Structure with Randomness: Probabilistic Algorithms for Constructing Approximate Matrix Decompositions". *SIAM Review*, 53(2), 217-288.

11. **Ross, D. A., Lim, J., Lin, R. S., & Yang, M. H.** (2008). "Incremental Learning for Robust Visual Tracking". *International Journal of Computer Vision*, 77(1), 125-141.

### Survey Papers

12. **Cunningham, J. P., & Ghahramani, Z.** (2015). "Linear Dimensionality Reduction: Survey, Insights, and Generalizations". *Journal of Machine Learning Research*, 16(1), 2859-2900.

---

## Summary

**Principal Component Analysis** is a foundational dimensionality reduction technique:

✓ **Strengths**:
  - Mathematically rigorous (variance maximization)
  - Computationally efficient (SVD)
  - Optimal linear approximation (Eckart-Young theorem)
  - Decorrelates features

✗ **Limitations**:
  - Linear method (fails on nonlinear manifolds)
  - Sensitive to scaling (requires standardization)
  - Interpretability challenges
  - Outlier sensitivity

**When to Use PCA**:
- High-dimensional data visualization
- Noise reduction in signals/images
- Feature engineering for downstream ML
- Data compression

**When to Avoid PCA**:
- Nonlinear structure dominates → Use Kernel PCA, autoencoders
- Small datasets (< 100 samples) → Overfitting risk
- Interpretability crucial → Use sparse methods or domain features

---